# CITYNEXUS — Colab Training Notebook

OpenEnv Hackathon 2026. Section 6 (GRPO) requires a Colab T4. Everything else is CPU.

## 0. Setup

In [ ]:
import os
GITHUB_URL = "https://github.com/YuvrajLamba01/CityNexus.git"
if not os.path.exists("CityNexus"):
    !git clone -q {GITHUB_URL}
%cd CityNexus

In [ ]:
!pip install -q -U pip
!pip install -q -e .
!pip install -q matplotlib numpy openenv-core

In [ ]:
import json, pathlib, statistics, random
import matplotlib.pyplot as plt
import numpy as np

from citynexus import (
    DeliveryAgent, TrafficAgent, EmergencyAgent, PoliceAgent, PlannerAgent,
    TrainingPipeline, TrainingConfig,
)
from citynexus.scenarios.generator import Curriculum
from citynexus.training.llm_planner import (
    MODES, obs_to_prompt, grpo_reward, build_dataset, expert_mode,
)
from server.environment import CityNexusEnvironment
from server.models import CityAction

os.makedirs("runs", exist_ok=True)

## 1. OpenEnv sanity check

In [ ]:
env = CityNexusEnvironment(max_ticks=20)
obs = env.reset(seed=11)
print(f"reset: tick={obs.tick} weather={obs.weather} done={obs.done}")
for mode in MODES:
    obs = env.step(CityAction(mode=mode, directive=f"mode:{mode}"))
    print(f"tick={obs.tick:>2} mode={mode:<16} reward={obs.reward:+.2f} city={obs.city_score:.2f}")
print(f"state: step_count={env.state.step_count} cumulative={env.state.cumulative_reward:+.2f}")

## 1b. Multi-agent inspection

In [ ]:
import copy as _copy
from citynexus.agents.coordinator import MultiAgentCoordinator, CoordinatorConfig
from citynexus.env.core import CityNexusEnv, EnvConfig
from citynexus.rewards.system import MultiAgentRewardSystem
from citynexus.verify.base import Verifier
from citynexus.verify.schemas import VerificationContext

env_mai = CityNexusEnv(EnvConfig(seed=7, max_ticks=10))
agents_mai = [DeliveryAgent(), TrafficAgent(), EmergencyAgent(), PoliceAgent(), PlannerAgent()]
coord_mai = MultiAgentCoordinator(env_mai, agents_mai, CoordinatorConfig(seed=7))
coord_mai.reset()
rsys_mai = MultiAgentRewardSystem(verifier=Verifier.default())
rsys_mai.reset()

prev_state = _copy.copy(env_mai.state)
res_mai = coord_mai.step()
vctx_mai = VerificationContext(
    tick=res_mai.step_info.tick, prev_state=prev_state, curr_state=env_mai.state,
    agent_ctx=coord_mai.ctx, actions=res_mai.actions,
    completed_deliveries=res_mai.completed_deliveries,
    new_deliveries=res_mai.new_deliveries,
    new_incidents=res_mai.new_incidents,
    accidents_cleared=res_mai.step_info.cleared_accidents,
    accidents_spawned=len(res_mai.step_info.new_accidents),
)
brk_mai = rsys_mai.compute(vctx_mai)

print(f"{'agent':<11} {'n_actions':>10} {'reward':>8}  components / penalties")
print("-" * 78)
for role, agent in coord_mai.agents.items():
    actions = res_mai.actions.get(role, [])
    par = brk_mai.per_agent.get(role.value)
    rew = par.total if par else 0.0
    parts = {}
    if par:
        parts.update(dict(par.components))
        parts.update({f"!{k}": v for k, v in par.penalties.items()})
    print(f"{role.value:<11} {len(actions):>10} {rew:>+8.2f}  {parts}")
print(f"\ncity_score={brk_mai.city_score.total:.3f}  summed={brk_mai.total_summed():+.3f}  gated={brk_mai.gated_any}")

## 2. Heuristic training pipeline

In [ ]:
def make_agents():
    return [DeliveryAgent(), TrafficAgent(), EmergencyAgent(), PoliceAgent(), PlannerAgent()]

config = TrainingConfig(
    n_episodes=40,
    max_ticks_per_episode=80,
    curriculum_target=0.55,
    curriculum_alpha=0.18,
    starting_difficulty=0.20,
    bias_top_k_modes=3,
    use_memory=True,
    memory_path="runs/memory.json",
    log_dir="runs",
    log_window=5,
    seed=42,
    console=True,
)
pipeline = TrainingPipeline(agents_factory=make_agents, config=config)
summary = pipeline.train()
print(f"episodes={summary.n_episodes} final_difficulty={summary.final_difficulty:.3f}")
print(f"mean_score={summary.mean_score:.3f} last5_score={summary.last_window_avg_score:.3f}")
print(f"mean_delivery={summary.mean_delivery_success:.3f} last5_delivery={summary.last_window_avg_success:.3f}")

## 3. Training curves

In [ ]:
records = pipeline.logger.all()
ep     = [r["episode"] for r in records]
score  = [r["score"] for r in records]
succ   = [r["delivery_success_rate"] for r in records]
diff   = [r["difficulty"] for r in records]
summed = [r["summed_reward"] for r in records]

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
ax[0, 0].plot(ep, score, label="score", alpha=0.7)
if len(score) >= 5:
    rolling = np.convolve(score, np.ones(5) / 5, mode="valid")
    ax[0, 0].plot(ep[4:], rolling, label="rolling-5", linewidth=2)
ax[0, 0].set(title="Episode score", xlabel="episode", ylabel="score"); ax[0, 0].grid(alpha=0.3); ax[0, 0].legend()
ax[0, 1].plot(ep, diff, color="tab:red")
ax[0, 1].set(title="Difficulty", xlabel="episode", ylabel="difficulty"); ax[0, 1].grid(alpha=0.3)
ax[1, 0].plot(ep, summed, color="tab:green")
ax[1, 0].set(title="Summed reward", xlabel="episode", ylabel="reward"); ax[1, 0].grid(alpha=0.3)
ax[1, 1].plot(ep, succ, color="tab:purple")
ax[1, 1].set(title="Delivery success", xlabel="episode", ylabel="rate"); ax[1, 1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("runs/training_curves.png", dpi=140, bbox_inches="tight")
plt.show()

## 3b. Per-agent reward attribution

In [ ]:
agent_totals = {}
for traj in pipeline.trajectories:
    for role, val in traj.cumulative_per_agent.items():
        agent_totals[role] = agent_totals.get(role, 0.0) + val

roles_order = ["delivery", "traffic", "emergency", "police", "planner"]
totals_a = [agent_totals.get(r, 0.0) for r in roles_order]
colors_a = ["#21d4fd", "#b721ff", "#fd9421", "#ff5e62", "#43e97b"]

fig, ax = plt.subplots(figsize=(9.5, 4.5))
bars = ax.bar(roles_order, totals_a, color=colors_a, edgecolor="black")
for b, v in zip(bars, totals_a):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:+.1f}",
            ha="center", va="bottom" if v >= 0 else "top", fontsize=10, fontweight="bold")
ax.axhline(0, color="black", linewidth=0.7)
ax.set_title("Cumulative reward by agent role (40 episodes)", fontweight="bold")
ax.set_ylabel("summed reward")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("runs/per_agent_rewards.png", dpi=140, bbox_inches="tight")
plt.show()
print("Per-agent cumulative rewards:")
for r in roles_order:
    print(f"  {r:<11} {agent_totals.get(r, 0.0):>+10.2f}")

## 4. Curriculum failure-mode replay

In [ ]:
cur = Curriculum(
    target_score=config.curriculum_target,
    alpha=config.curriculum_alpha,
    starting_difficulty=config.starting_difficulty,
)
names = ["delivery_failure", "accident_pileup", "incident_pileup", "congestion_overload"]
series = {n: [] for n in names}
diffs = []
metrics = [t.final_metrics for t in pipeline.trajectories if t.final_metrics]
for m in metrics:
    cur.update(m)
    for n in names:
        series[n].append(round(cur._failure_ema.get(n, 0.0), 4))
    diffs.append(round(cur.difficulty, 4))

x = list(range(1, len(metrics) + 1))
palette = {"delivery_failure": "tab:blue", "accident_pileup": "tab:red",
           "incident_pileup": "tab:orange", "congestion_overload": "tab:green"}
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for n in names:
    if any(v > 0.0 for v in series[n]):
        ax[0].plot(x, series[n], label=n, color=palette[n], linewidth=2)
ax[0].set(title="Curriculum failure-mode EMA", ylabel="EMA severity"); ax[0].legend(loc="upper left"); ax[0].grid(alpha=0.3)
ax[1].plot(x, diffs, color="tab:red", linewidth=2.2)
ax[1].set(title="Adaptive difficulty", xlabel="episode", ylabel="difficulty"); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("runs/curriculum_failure_modes.png", dpi=140, bbox_inches="tight")
plt.show()

## 4b. Persistent memory snapshot

In [ ]:
mem = pipeline.memory
if mem is None:
    print("memory disabled in TrainingConfig")
else:
    s = mem.stats()
    print(f"total_records = {s['total']}")
    print(f"by_kind       = {s['by_kind']}")
    failures = mem.common_failure_modes(top_n=5)
    if failures:
        print("\nTop failure modes (count):")
        for mode, n in failures:
            print(f"  {mode:<25} {n}")
    zones = mem.hottest_zones(top_k=5)
    if zones:
        print("\nHottest risk zones:")
        for z in zones:
            print(f"  id={z.id} risk={z.risk_score:.2f} coords={list(z.coords)[:3]}")
    print(f"\npersisted to: {s['path']}")

## 5. Baseline vs trained

In [ ]:
rows = [json.loads(x) for x in pathlib.Path("runs/training.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
base_score    = rows[0]["score"]
trained_score = statistics.fmean(r["score"] for r in rows[-5:])
base_succ     = rows[0]["delivery_success_rate"]
trained_succ  = statistics.fmean(r["delivery_success_rate"] for r in rows[-5:])

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for axis, m, t in ((ax[0], [base_score, trained_score], "Score"),
                   (ax[1], [base_succ,  trained_succ],  "Delivery success")):
    bars = axis.bar(["baseline", "trained"], m, color=["#7a7a92", "#21d4fd"])
    axis.set_title(t); axis.set_ylim(0, 1); axis.grid(alpha=0.3, axis="y")
    for b, v in zip(bars, m):
        axis.text(b.get_x() + b.get_width() / 2, v + 0.03, f"{v:.2f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("runs/baseline_vs_trained.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"baseline_score={base_score:.3f} trained_score={trained_score:.3f}")
print(f"baseline_success={base_succ:.3f} trained_success={trained_succ:.3f}")

## 5b. Heuristic Planner action distribution

In [ ]:
from collections import Counter

mode_counts = Counter()
for s in range(2000, 2010):
    e = CityNexusEnvironment(max_ticks=80); o = e.reset(seed=s)
    while not o.done:
        m = expert_mode(o.model_dump())
        mode_counts[m] += 1
        o = e.step(CityAction(mode=m))

modes_h = list(MODES)
counts_h = [mode_counts.get(m, 0) for m in modes_h]
total_h = sum(counts_h) or 1

fig, ax = plt.subplots(figsize=(9.5, 4.2))
bars = ax.bar(modes_h, counts_h, color=["#7a7a92", "#ff5e62", "#21d4fd", "#43e97b"], edgecolor="black")
for b, c in zip(bars, counts_h):
    ax.text(b.get_x() + b.get_width() / 2, c, f"{c}\n({100 * c / total_h:.0f}%)",
            ha="center", va="bottom", fontsize=9)
ax.set_title("Heuristic Planner: mode distribution (10 seeds x 80 ticks)", fontweight="bold")
ax.set_ylabel("ticks chosen")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("runs/heuristic_action_dist.png", dpi=140, bbox_inches="tight")
plt.show()
print(f"unique_modes_used = {sum(1 for c in counts_h if c > 0)}/{len(MODES)}  total_ticks = {sum(counts_h)}")

## 6. GRPO LLM-planner (T4 GPU)

In [ ]:
!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.27"
!pip install -q trl peft accelerate bitsandbytes datasets

In [ ]:
# Build the dataset with env-driven rewards. For each observation we replay
# the env four times (once per mode) and record the next-tick reward — this
# is what `reward_env_lookahead` scores GRPO completions against, so the LLM
# can learn modes the hand-coded heuristic gets wrong.
samples = build_dataset(
    CityNexusEnvironment(max_ticks=80),
    n_episodes=40, seed=0, base_seed=1000,
    action_cls=CityAction,
    env_factory=lambda: CityNexusEnvironment(max_ticks=80),
    with_env_rewards=True,
)
dist = {m: sum(1 for s in samples if s.expert == m) for m in MODES}
print(f"samples={len(samples)} expert_distribution={dist}")

# Quick sanity: how often does the heuristic disagree with the env-best mode?
import statistics as _stats
def _best_env_mode(s):
    return max(s.rewards_by_mode, key=s.rewards_by_mode.get)
matches = [_best_env_mode(s) == s.expert for s in samples if s.rewards_by_mode]
match_rate = _stats.mean([1.0 if m else 0.0 for m in matches]) if matches else 0.0
print(f"heuristic ↔ env-best agreement: {match_rate:.1%}  "
      f"(the env-driven reward unlocks the remaining {1 - match_rate:.0%}).")


In [ ]:
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

# Four independent reward components — TRL logs each separately so judges can
# read per-component curves (hackathon guide §7). The env-driven signal is
# what lets the model exceed the heuristic ceiling.
from citynexus.training.llm_planner import (
    reward_correctness, reward_format, reward_length, reward_env_lookahead,
)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=512, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=8, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

def chat(p):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True,
    )

# `rewards_by_mode` is a dict per row — TRL passes it through to the reward
# function as a list of dicts (one per completion).
ds = Dataset.from_list([
    {
        "prompt": chat(s.prompt),
        "expert": s.expert,
        "rewards_by_mode": dict(s.rewards_by_mode) if s.rewards_by_mode else {},
    }
    for s in samples
])

cfg = GRPOConfig(
    output_dir="runs/grpo",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=5,
    max_prompt_length=400,
    max_completion_length=8,
    num_generations=4,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    seed=42,
)
# Reward function order matters for TRL's logging keys but not for the
# advantage computation (TRL averages all functions equally by default).
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[
        reward_correctness,    # match heuristic-best mode (bootstrapping)
        reward_format,         # well-formed mode name (anti-gaming)
        reward_length,         # discourage essay completions
        reward_env_lookahead,  # actual env outcome (exceeds heuristic ceiling)
    ],
    args=cfg,
    train_dataset=ds,
)
trainer.train()


In [ ]:
import matplotlib.pyplot as plt

# TRL logs each reward function under rewards/<func_name>/mean in log_history.
# Plot all four components plus the combined mean.
log = trainer.state.log_history

def _series(key):
    pts = [(h.get("step"), h[key]) for h in log if key in h]
    return [p[0] for p in pts], [p[1] for p in pts]

steps_c, r_correct  = _series("rewards/reward_correctness/mean")
steps_f, r_format   = _series("rewards/reward_format/mean")
steps_l, r_length   = _series("rewards/reward_length/mean")
steps_e, r_envlook  = _series("rewards/reward_env_lookahead/mean")
steps_m, r_mean     = _series("reward")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(steps_c, r_correct, label="correctness", color="#21d4fd", linewidth=2)
axes[0].plot(steps_f, r_format,  label="format",      color="#10b981", linewidth=2)
axes[0].plot(steps_l, r_length,  label="length",      color="#f59e0b", linewidth=2)
axes[0].plot(steps_e, r_envlook, label="env_lookahead (RLVR)", color="#ff3b6b", linewidth=2)
axes[0].set_title("GRPO reward components per step", fontweight="bold")
axes[0].set_xlabel("step"); axes[0].set_ylabel("mean reward")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(steps_m, r_mean, color="#7c5cff", linewidth=2)
axes[1].set_title("GRPO combined mean reward", fontweight="bold")
axes[1].set_xlabel("step"); axes[1].set_ylabel("reward")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("runs/grpo_reward.png", dpi=140, bbox_inches="tight")
plt.show()
print("saved runs/grpo_reward.png — 4 component curves + combined mean")


In [ ]:
FastLanguageModel.for_inference(model)

def llm_pick_mode(obs_dict):
    inp = tokenizer(chat(obs_to_prompt(obs_dict)), return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=8, do_sample=False, temperature=0.0)
    text = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    for m in MODES:
        if m in text:
            return m
    return "normal"

trained_scores, random_scores = [], []
trained_modes, random_modes = [], []
trained_modes_by_seed, random_modes_by_seed = {}, {}
trained_cum_by_seed,  random_cum_by_seed  = {}, {}
for seed in range(3000, 3010):
    e = CityNexusEnvironment(max_ticks=80); o = e.reset(seed=seed); cum = 0.0
    seed_modes_t = []
    while not o.done:
        m = llm_pick_mode(o.model_dump())
        seed_modes_t.append(m); trained_modes.append(m)
        o = e.step(CityAction(mode=m)); cum += o.reward
    trained_scores.append(cum)
    trained_modes_by_seed[str(seed)] = seed_modes_t
    trained_cum_by_seed[str(seed)]   = round(cum, 4)
    e = CityNexusEnvironment(max_ticks=80); o = e.reset(seed=seed); cum = 0.0; rng = random.Random(seed)
    seed_modes_r = []
    while not o.done:
        m = rng.choice(MODES)
        seed_modes_r.append(m); random_modes.append(m)
        o = e.step(CityAction(mode=m)); cum += o.reward
    random_scores.append(cum)
    random_modes_by_seed[str(seed)] = seed_modes_r
    random_cum_by_seed[str(seed)]   = round(cum, 4)

rmean, rstd = statistics.mean(random_scores),  statistics.stdev(random_scores)
tmean, tstd = statistics.mean(trained_scores), statistics.stdev(trained_scores)
delta = tmean - rmean
pct = 100.0 * delta / abs(rmean) if rmean else 0.0
print(f"random={rmean:+.2f}±{rstd:.2f} trained={tmean:+.2f}±{tstd:.2f} delta={delta:+.2f} ({pct:+.1f}%)")

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(["random", "GRPO-trained"], [rmean, tmean], yerr=[rstd, tstd],
              color=["#7a7a92", "#21d4fd"], edgecolor="black", capsize=10)
for b, v in zip(bars, [rmean, tmean]):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:+.2f}", ha="center", va="bottom")
ax.set_title("Cumulative env reward, 10 held-out seeds", fontweight="bold")
ax.set_ylabel("cumulative reward"); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("runs/llm_vs_random.png", dpi=140, bbox_inches="tight")
plt.show()

## 6b. Statistical significance + mode distribution

In [ ]:
import math
from collections import Counter

def _welch(a, b):
    ma, mb = statistics.mean(a), statistics.mean(b)
    va, vb = statistics.variance(a), statistics.variance(b)
    na, nb = len(a), len(b)
    se = math.sqrt(va / na + vb / nb)
    t = (mb - ma) / se if se > 0 else float("inf")
    df_num = (va / na + vb / nb) ** 2
    df_den = (va / na) ** 2 / (na - 1) + (vb / nb) ** 2 / (nb - 1)
    df = df_num / df_den if df_den > 0 else float("inf")
    return t, df

def _two_tailed_p(t, df):
    z = abs(t) * (1 - 1 / (4 * df)) / math.sqrt(1 + t * t / (2 * df))
    return 2 * (1 - 0.5 * (1 + math.erf(z / math.sqrt(2))))

t_stat, df = _welch(random_scores, trained_scores)
p_val = _two_tailed_p(t_stat, df)
pooled = math.sqrt((statistics.variance(random_scores) + statistics.variance(trained_scores)) / 2)
cohens_d = (statistics.mean(trained_scores) - statistics.mean(random_scores)) / pooled
effect = "large" if abs(cohens_d) >= 0.8 else "medium" if abs(cohens_d) >= 0.5 else "small"

print(f"Welch's t-test (random vs GRPO-trained, n=10):")
print(f"  t  = {t_stat:+.3f}   df = {df:.1f}   p (two-tailed) = {p_val:.4f}")
print(f"  Cohen's d = {cohens_d:+.3f}  ({effect} effect)")
print(f"  verdict: {'SIGNIFICANT (p < 0.05)' if p_val < 0.05 else 'not significant at alpha=0.05'}")

trained_dist = Counter(trained_modes)
random_dist  = Counter(random_modes)
mt = [trained_dist.get(m, 0) for m in MODES]
mr = [random_dist.get(m, 0)  for m in MODES]

fig, ax = plt.subplots(figsize=(10.5, 4.5))
x = np.arange(len(MODES))
ax.bar(x - 0.18, mr, 0.36, label="random",       color="#7a7a92", edgecolor="black")
ax.bar(x + 0.18, mt, 0.36, label="GRPO-trained", color="#21d4fd", edgecolor="black")
for xi, vr, vt in zip(x, mr, mt):
    ax.text(xi - 0.18, vr, str(vr), ha="center", va="bottom", fontsize=9)
    ax.text(xi + 0.18, vt, str(vt), ha="center", va="bottom", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(MODES)
ax.set_ylabel("ticks chosen"); ax.legend()
ax.set_title("Mode distribution: random vs GRPO-trained (10 seeds x 80 ticks)", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("runs/llm_mode_distribution.png", dpi=140, bbox_inches="tight")
plt.show()

## 6c. Static-demo rollouts dump (powers the in-browser playback panel)

Append the trained-LLM mode trace to `runs/llm_rollouts.json` and `web/data/llm_rollouts.json`.
The static demo Space loads this file so a reviewer can click `Play recorded policy` and
watch the GRPO-trained policy's per-tick mode choices animate the in-browser city —
side by side with the random baseline and the heuristic expert.

In [ ]:
from datetime import datetime, timezone

ROLLOUTS_PATHS = ["runs/llm_rollouts.json", "web/data/llm_rollouts.json"]

base_payload = json.loads(pathlib.Path(ROLLOUTS_PATHS[0]).read_text(encoding="utf-8"))

base_payload["tracks"]["trained_llm"] = {
    "label": "GRPO-trained Qwen-2.5-0.5B Planner",
    "description": (
        "Per-tick mode chosen by the trained LLM during evaluation on 10 held-out "
        "seeds (3000-3009) at max_ticks=80. Generated by Section 6c of the Colab "
        "notebook after Section 6 finishes."
    ),
    "available": True,
    "rollouts": {
        seed: {
            "modes": trained_modes_by_seed[seed],
            "cumulative_reward": trained_cum_by_seed[seed],
        }
        for seed in trained_modes_by_seed
    },
    "summary": {
        "n_seeds": len(trained_scores),
        "mean_cumulative_reward": round(statistics.mean(trained_scores), 4),
        "stdev_cumulative_reward": round(statistics.stdev(trained_scores), 4),
        "delta_vs_random": round(statistics.mean(trained_scores) - statistics.mean(random_scores), 4),
        "welch_t": round(t_stat, 4),
        "p_value": round(p_val, 6),
        "cohens_d": round(cohens_d, 4),
    },
}

base_payload["tracks"]["random_baseline"]["rollouts"] = {
    seed: {
        "modes": random_modes_by_seed[seed],
        "cumulative_reward": random_cum_by_seed[seed],
    }
    for seed in random_modes_by_seed
}
base_payload["generated_at"] = datetime.now(tz=timezone.utc).isoformat()

for path in ROLLOUTS_PATHS:
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(base_payload, indent=2), encoding="utf-8")
    print(f"wrote {path}  ({p.stat().st_size // 1024} KB)")

print(
    f"trained_llm track: mean={statistics.mean(trained_scores):+.2f}, "
    f"random_baseline track: mean={statistics.mean(random_scores):+.2f}"
)

## 6d. (Optional) Push trained model to Hugging Face Hub

Lets the docker OpenEnv Space, the playback Space, or any external evaluator
load the trained Planner with `from_pretrained(repo_id)`. Skip this cell if
you only need the local artifacts.

Set `HF_TOKEN` in Colab Secrets (Settings -> Secrets) before running.

In [ ]:
REPO_ID = "YuvrajLamba01/citynexus-planner-qwen2.5-0.5b"

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    print("HF_TOKEN not set; skipping push. Set it in Colab Secrets to enable.")
else:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    model.push_to_hub_merged(REPO_ID, tokenizer, save_method="merged_16bit", token=hf_token)
    print(f"pushed merged model to https://huggingface.co/{REPO_ID}")
    print("Set CITYNEXUS_LLM_REPO=" + REPO_ID + " on the docker Space to serve it.")

## 7. Reward-hacking probe

In [ ]:
completions = [
    "normal",
    "normal but here is some extra explanation that runs long",
    "defensive",
    "qqq totally invalid output \u00b6\u00b6\u00b6",
]
rewards_p = grpo_reward(None, completions, ["normal"] * 4)
for c, r in zip(completions, rewards_p):
    print(f"{r:+.2f}  {c[:60]}")
assert rewards_p[0] > rewards_p[1] > rewards_p[2] > rewards_p[3]

## 8. Evidence + headline metrics

In [ ]:
for f in [
    "runs/training.jsonl",
    "runs/memory.json",
    "runs/training_curves.png",
    "runs/per_agent_rewards.png",
    "runs/curriculum_failure_modes.png",
    "runs/baseline_vs_trained.png",
    "runs/heuristic_action_dist.png",
    "runs/grpo_reward.png",
    "runs/llm_vs_random.png",
    "runs/llm_mode_distribution.png",
    "runs/llm_rollouts.json",
    "web/data/llm_rollouts.json",
]:
    p = pathlib.Path(f)
    flag = "OK" if p.exists() else "MISSING"
    size = f"{p.stat().st_size // 1024} KB" if p.exists() else "-"
    print(f"  [{flag:<7}] {f:<40} {size}")

In [ ]:
rows = [json.loads(x) for x in pathlib.Path("runs/training.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
last5 = rows[-5:] if len(rows) >= 5 else rows
print("### Heuristic Pipeline (Sections 2-5)")
print(f"- Episodes trained:               {len(rows)}")
print(f"- Mean score (all):               {statistics.fmean(r['score'] for r in rows):.3f}")
print(f"- Last-5 avg score:               {statistics.fmean(r['score'] for r in last5):.3f}")
print(f"- Mean delivery success:          {statistics.fmean(r['delivery_success_rate'] for r in rows):.3f}")
print(f"- Last-5 delivery success:        {statistics.fmean(r['delivery_success_rate'] for r in last5):.3f}")
agent_sum = {}
for r in rows:
    for k, v in (r.get("cumulative_per_agent") or {}).items():
        agent_sum[k] = agent_sum.get(k, 0.0) + v
if agent_sum:
    print(f"- Per-agent cumulative reward:    " + ", ".join(f"{k}={v:+.1f}" for k, v in agent_sum.items()))
print()
try:
    print("### GRPO LLM RL (Section 6)")
    rmean = statistics.mean(random_scores);  rstd = statistics.stdev(random_scores)
    tmean = statistics.mean(trained_scores); tstd = statistics.stdev(trained_scores)
    delta = tmean - rmean
    pct = 100.0 * delta / abs(rmean) if rmean else 0.0
    print(f"- Random-mode baseline cumulative reward: mean={rmean:+.2f}, stdev={rstd:.2f}")
    print(f"- GRPO-trained LLM cumulative reward:     mean={tmean:+.2f}, stdev={tstd:.2f}")
    print(f"- Improvement (trained - baseline):       {delta:+.2f}  ({pct:+.1f}%)")
    try:
        print(f"- Welch's t-test:                         t={t_stat:+.3f}, df={df:.1f}, p={p_val:.4f}")
        print(f"- Cohen's d:                              {cohens_d:+.3f}  ({effect} effect)")
    except NameError:
        pass
except NameError:
    print("### GRPO LLM RL (Section 6) - not run in this session")